# Simulate Traffic — Peupler `prediction_logs`

2 batchs envoyés à l'API :
- **neutre** : sample aléatoire du dataset d'entraînement (pas de drift attendu)
- **drift_simule** : sample filtré (EXT_SOURCE_2 bas + AMT_INCOME_TOTAL haut) pour simuler une dérive de population

In [23]:
import pandas as pd
import requests
import time

API_URL = "https://jackobess-oc-projet8.hf.space/"  # URL API (Hugging Face Space)
PREDICT_ENDPOINT = f"{API_URL}/predict"
HEALTH_ENDPOINT = f"{API_URL}/health" 

N_NEUTRE = 30       # nombre de requêtes neutres (sans drift)
N_DRIFT = 30        # nombre de requêtes avec drift

PARQUET_PATH = r"C:\Users\jacob\Desktop\Dev\OpenClassRoom\Projet-6-Initiez-vous-au-MLOps-1\input\data_clean.parquet"  # Chemin vers le fichier Parquet

## 1. Health check

On vérifie que l'API répond

In [24]:
try:
    resp = requests.get(HEALTH_ENDPOINT, timeout=10)
    print(f"Status: {resp.status_code}")
    print(resp.text[:300])
    assert resp.status_code == 200, "API non dispo !!"
except requests.RequestException as e:
    raise SystemExit(f"API injoignable : {e}")

Status: 200
{"status":"ok","version":"1.1.0","environment":"HF_Prod"}


## 2. Chargement des données d'entraînement

In [25]:
df = pd.read_parquet(PARQUET_PATH)
cols_to_drop = [c for c in ["SK_ID_CURR", "TARGET"] if c in df.columns]
df = df.drop(columns=cols_to_drop)

print(df.shape)
df.head(5)

(356251, 795)


,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,...,CC_NAME_CONTRACT_STATUS_Signed_MAX,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_Signed_SUM,CC_NAME_CONTRACT_STATUS_Signed_VAR,CC_NAME_CONTRACT_STATUS_nan_MIN,CC_NAME_CONTRACT_STATUS_nan_MAX,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_NAME_CONTRACT_STATUS_nan_SUM,CC_NAME_CONTRACT_STATUS_nan_VAR,CC_COUNT
0,0,0,0,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0,1,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,1,0,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,0,0,0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0
4,0,0,0,0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Sampling neutre vs drift simulé

In [26]:
df_neutre = df.sample(n=N_NEUTRE, random_state=42)
# Médianes calculées sur tout le dataset -> sert à combler les NaN
# (LightGBM gère les NaN nativement à l'entraînement, mais l'API rejette les null en JSON)
medians = df.median(numeric_only=True).to_dict()

df_filtered = df.copy()
if "EXT_SOURCE_2" in df_filtered.columns:
    df_filtered = df_filtered[df_filtered["EXT_SOURCE_2"] < df_filtered["EXT_SOURCE_2"].quantile(0.25)]
if "AMT_INCOME_TOTAL" in df_filtered.columns:
    df_filtered = df_filtered[df_filtered["AMT_INCOME_TOTAL"] > df_filtered["AMT_INCOME_TOTAL"].quantile(0.75)]  # haut revenus
if "DAYS_BIRTH" in df_filtered.columns:
    df_filtered = df_filtered[df_filtered["DAYS_BIRTH"] > df_filtered["DAYS_BIRTH"].quantile(0.75)]  # jeunes

df_drift = df_filtered.sample(n=min(N_DRIFT, len(df_filtered)), random_state=42)

print(f"Batch neutre : {df_neutre.shape}")
print(f"Batch drift  : {df_drift.shape}")

Batch neutre : (30, 795)
Batch drift  : (30, 795)


In [27]:
# Comparaison rapide des distributions pour vérifier que le filtre a bien un effet
cols_check = [c for c in ["EXT_SOURCE_2", "AMT_INCOME_TOTAL", "DAYS_BIRTH"] if c in df.columns]
pd.DataFrame({
    "neutre_mean": df_neutre[cols_check].mean(),
    "drift_mean": df_drift[cols_check].mean(),
    "full_mean": df[cols_check].mean(),
})

,neutre_mean,drift_mean,full_mean
EXT_SOURCE_2,0.474341,0.235529,0.514889
AMT_INCOME_TOTAL,174320.250000,276303.900000,170115.873094
DAYS_BIRTH,-17251.366667,-9903.333333,-16041.276687


## 4. Envoi des requêtes à l'API

In [29]:
def send_requests(df_sample, label, medians):
    print(f"\n--- Envoi batch '{label}' ({len(df_sample)} lignes) ---")
    results = []

    for i, row in df_sample.iterrows():
        raw = row.to_dict()
        raw = {k: (medians.get(k, 0) if pd.isna(v) else v) for k, v in raw.items()}
        payload = {"features": raw}

        try:
            resp = requests.post(PREDICT_ENDPOINT, json=payload, timeout=15)
            ok = resp.status_code == 200
            results.append({"index": i, "status_code": resp.status_code, "ok": ok})
            if not ok:
                print(f"  [{i}] HTTP {resp.status_code}: {resp.text[:150]}")
        except requests.RequestException as e:
            results.append({"index": i, "status_code": None, "ok": False})
            print(f"  [{i}] Exception: {e}")

        time.sleep(0.2)

    df_results = pd.DataFrame(results)
    n_success = df_results["ok"].sum()
    print(f"'{label}' terminé : {n_success} success / {len(df_results) - n_success} erreurs")
    return df_results

In [31]:
results_neutre = send_requests(df_neutre, "neutre", medians)
results_neutre


--- Envoi batch 'neutre' (30 lignes) ---
'neutre' terminé : 30 success / 0 erreurs


,index,status_code,ok
0,24226,200,True
1,329008,200,True
2,205316,200,True
3,162010,200,True
4,71831,200,True
5,108932,200,True
6,18323,200,True
7,199850,200,True
8,12624,200,True
9,327462,200,True


In [32]:
results_drift = send_requests(df_drift, "drift_simule", medians)
results_drift


--- Envoi batch 'drift_simule' (30 lignes) ---
'drift_simule' terminé : 30 success / 0 erreurs


,index,status_code,ok
0,27702,200,True
1,355986,200,True
2,112851,200,True
3,256148,200,True
4,242843,200,True
5,66258,200,True
6,187741,200,True
7,54445,200,True
8,278136,200,True
9,308870,200,True


## 5. Récap

Va checker `prediction_logs` sur Neon, puis enchaîne sur `drift_analysis.ipynb`.

In [33]:
summary = pd.concat([
    results_neutre.assign(batch="neutre"),
    results_drift.assign(batch="drift_simule"),
])
summary.groupby("batch")["ok"].agg(["sum", "count"])

,sum,count
batch,,
drift_simule,30,30
neutre,30,30
